# Decision-Aware & Explainable Deep Learning for Leukemic Blast Detection
**Clean reference implementation (final).**

This notebook implements the full methodology of the paper:
patient-independent split, confidence-based pseudo-labeling, a dual-head
ResNet-18 (classification + decision), a calibrated ACCEPT/REVIEW/DEFER policy,
selective-prediction evaluation with patient-level bootstrap CIs, ECE calibration,
matched-protocol baselines (Confidence Thresholding, MC-Dropout, Deep Ensemble),
decision-level Grad-CAM attention metrics, a faithfulness check, and automatic
figure generation.

> All metrics are computed from real model outputs. Nothing is hardcoded.
> Run on the C-NMC data and report exactly what `RESULTS_SUMMARY.json` contains.


## 1. Setup and configuration
Edit `DATA_DIR` and `OUT_DIR` to your environment. `DATA_DIR` must contain `blast/` and `normal/` subfolders of `.bmp` images whose filenames keep the C-NMC `UID_<patient>_...` token.

In [ ]:
import os, re, json, math, random, copy, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

# ----- reproducibility -----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----- paths (EDIT THESE) -----
DATA_DIR = "/content/drive/MyDrive/C-NMC_training_data"
OUT_DIR  = "/content/drive/MyDrive/hematology_project_outputs"
FIG_DIR  = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

# ----- hyper-parameters (final settings) -----
IMG_SIZE, BATCH_SIZE = 224, 32
LR, WEIGHT_DECAY     = 5e-5, 1e-5
MAX_EPOCHS, PATIENCE = 60, 15
ALPHA, BETA          = 1.0, 0.7          # joint-loss weights
THETA_A, THETA_R, DELTA = 0.75, 0.60, 0.15
TARGET_COVERAGE      = 0.70              # match baselines at this coverage
DEFER_VAR_PERCENTILE = 2.0
ACCEPT_TOP_FRAC, REVIEW_BOTTOM_FRAC, DEFER_LABEL_FRAC = 0.60, 0.20, 0.02
N_BOOTSTRAP = 1000
ACCEPT, REVIEW, DEFER = 0, 1, 2
print("Device:", DEVICE)

## 2. (Colab) Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    # Optional speed-up: copy data to local disk for faster repeated reads
    import shutil
    LOCAL = "/content/cnmc_local"
    if os.path.isdir(DATA_DIR) and not os.path.isdir(LOCAL):
        print("Copying data to local disk (one-time)...")
        shutil.copytree(DATA_DIR, LOCAL); DATA_DIR = LOCAL
    elif os.path.isdir(LOCAL):
        DATA_DIR = LOCAL
    print("DATA_DIR =", DATA_DIR)
except Exception as e:
    print("Not on Colab or Drive not used:", e)

## 3. Transforms

In [ ]:
NORM_MEAN, NORM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])

## 4. Patient-independent split
Patient id is parsed from the filename (`UID_48_...`→`48`, `UID_H12_...`→`H12`). Splitting is done **per patient within each class**, guaranteeing both classes appear in every partition and no patient crosses partitions.

In [ ]:
PATIENT_RE = re.compile(r"UID[_-]?([A-Za-z]?\d+)", re.IGNORECASE)
def patient_id_from_path(path):
    m = PATIENT_RE.search(os.path.basename(path))
    return m.group(1) if m else os.path.basename(path).split("_")[0]

def build_patient_split(data_dir):
    base = datasets.ImageFolder(data_dir)
    classes, samples = base.classes, base.samples
    targets = np.array([y for _, y in samples])
    groups  = np.array([patient_id_from_path(p) for p, _ in samples])
    idx = np.arange(len(samples))
    print(f"Classes: {classes} | images: {len(samples)} | unique patients: {len(set(groups))}")

    pat_cls = {p: int(np.bincount(targets[groups == p]).argmax()) for p in np.unique(groups)}
    rng = np.random.default_rng(SEED)
    tr, va, te = [], [], []
    for c in np.unique(list(pat_cls.values())):
        pats = np.array([p for p in pat_cls if pat_cls[p] == c]); rng.shuffle(pats)
        n = len(pats); n_tr = max(int(round(.70*n)), 1); n_va = max(int(round(.15*n)), 1) if n >= 3 else 0
        tr_p, va_p, te_p = pats[:n_tr], pats[n_tr:n_tr+n_va], pats[n_tr+n_va:]
        if len(te_p) == 0 and n >= 2: te_p, va_p = va_p[-1:], va_p[:-1]
        for p in tr_p: tr += idx[groups == p].tolist()
        for p in va_p: va += idx[groups == p].tolist()
        for p in te_p: te += idx[groups == p].tolist()
    tr, va, te = map(np.array, (tr, va, te))
    g = lambda ii: set(groups[ii].tolist())
    assert g(tr).isdisjoint(g(va)) and g(tr).isdisjoint(g(te)) and g(va).isdisjoint(g(te))
    def rep(name, ii):
        b = int((targets[ii] == classes.index('blast')).sum())
        print(f"  {name:5s}: images={len(ii):5d} | patients={len(g(ii)):3d} | blast={b} normal={len(ii)-b}")
    rep("train", tr); rep("val", va); rep("test", te)
    json.dump({"classes":classes,"train":tr.tolist(),"val":va.tolist(),"test":te.tolist(),
               "patients":{"train":sorted(g(tr)),"val":sorted(g(va)),"test":sorted(g(te))}},
              open(os.path.join(OUT_DIR,"patient_split.json"),"w"))
    return base, classes, samples, targets, groups, tr, va, te

def make_loaders(d, tr, va, te):
    return (DataLoader(Subset(datasets.ImageFolder(d, transform=train_tfms), list(tr)), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(Subset(datasets.ImageFolder(d, transform=eval_tfms),  list(va)), BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True),
            DataLoader(Subset(datasets.ImageFolder(d, transform=eval_tfms),  list(te)), BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True))

## 5. Models and training
A baseline ResNet-18 generates decision pseudo-labels; the dual-head model shares a backbone and adds a 3-way decision head. Training uses Adam, gradient clipping, and early stopping. The decision loss safely ignores the unlabeled middle band.

In [ ]:
def make_resnet18(n_classes=2, dropout=0.0):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(f, n_classes)) if dropout > 0 else nn.Linear(f, n_classes)
    return m.to(DEVICE)

def train_classifier(model, tr, va, cw=None, tag="clf"):
    crit = nn.CrossEntropyLoss(weight=cw.to(DEVICE) if cw is not None else None)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best, best_state, bad = math.inf, None, 0
    for ep in range(1, MAX_EPOCHS+1):
        model.train()
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y)
            if torch.isnan(loss): continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for x, y in va:
                x, y = x.to(DEVICE), y.to(DEVICE); vl += crit(model(x), y).item()*x.size(0); n += x.size(0)
        vl /= max(n,1)
        if vl < best-1e-4: best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        print(f"[{tag}] epoch {ep:3d} | val_loss {vl:.4f} | best {best:.4f} | bad {bad}")
        if bad >= PATIENCE: print(f"[{tag}] early stop @ {ep}"); break
    if best_state: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_logits(model, loader):
    model.eval(); L, Y = [], []
    for x, y in loader: L.append(model(x.to(DEVICE)).cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

class DualHeadNet(nn.Module):
    def __init__(self, n_classes=2, n_actions=3):
        super().__init__()
        bk = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        f = bk.fc.in_features; bk.fc = nn.Identity()
        self.backbone, self.cls_head, self.dec_head = bk, nn.Linear(f, n_classes), nn.Linear(f, n_actions)
    def forward(self, x):
        h = self.backbone(x); return self.cls_head(h), self.dec_head(h)

class IndexedSubset(torch.utils.data.Dataset):
    def __init__(self, d, indices, tfms):
        self.ds = datasets.ImageFolder(d, transform=tfms); self.indices = list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        gi = self.indices[i]; x, y = self.ds[gi]; return x, y, gi

## 6. Confidence-based decision pseudo-labels

In [ ]:
@torch.no_grad()
def intensity_variance(loader):
    mean = torch.tensor(NORM_MEAN).view(1,3,1,1); std = torch.tensor(NORM_STD).view(1,3,1,1)
    return np.concatenate([((x*std+mean).var(dim=(1,2,3))).numpy() for x,_ in loader])

def build_decision_labels(loader, baseline):
    logits,_ = predict_logits(baseline, loader)
    conf = logits.softmax(1).max(1).values.numpy(); var = intensity_variance(loader)
    dec = np.full(len(conf), -1, np.int64); sup = np.zeros(len(conf), bool)
    dmask = var <= np.percentile(var, DEFER_LABEL_FRAC*100); dec[dmask]=DEFER; sup[dmask]=True
    elig = ~dmask
    amask = elig & (conf >= np.quantile(conf[elig], 1-ACCEPT_TOP_FRAC)); dec[amask]=ACCEPT; sup[amask]=True
    rmask = elig & ~amask & (conf <= np.quantile(conf[elig], REVIEW_BOTTOM_FRAC)); dec[rmask]=REVIEW; sup[rmask]=True
    print(f"Pseudo-labels -> ACCEPT {amask.sum()} | REVIEW {rmask.sum()} | DEFER {dmask.sum()} | ignored {(~sup).sum()}")
    return torch.tensor(dec), torch.tensor(sup)

def train_dual_head(d, tr_i, va_i, dec_labels, cw=None):
    tr = DataLoader(IndexedSubset(d, tr_i, train_tfms), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    va = DataLoader(IndexedSubset(d, va_i, eval_tfms),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    model = DualHeadNet().to(DEVICE)
    cls_crit = nn.CrossEntropyLoss(weight=cw.to(DEVICE) if cw is not None else None)
    def dec_crit(logits, dd):
        m = dd != -1
        return logits.sum()*0.0 if m.sum()==0 else F.cross_entropy(logits[m], dd[m])
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    dec_labels = dec_labels.to(DEVICE)
    best, best_state, bad = math.inf, None, 0
    for ep in range(1, MAX_EPOCHS+1):
        model.train()
        for x, y, gi in tr:
            x, y = x.to(DEVICE), y.to(DEVICE); dd = dec_labels[gi.to(DEVICE)]
            cl, dl = model(x); loss = ALPHA*cls_crit(cl,y) + BETA*dec_crit(dl,dd)
            if torch.isnan(loss) or torch.isinf(loss): continue
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for x, y, gi in va:
                x, y = x.to(DEVICE), y.to(DEVICE); dd = dec_labels[gi.to(DEVICE)]
                cl, dl = model(x); _l = ALPHA*cls_crit(cl,y)+BETA*dec_crit(dl,dd)
                if torch.isnan(_l) or torch.isinf(_l): continue
                vl += _l.item()*x.size(0); n += x.size(0)
        vl /= max(n,1)
        if vl < best-1e-4: best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        print(f"[dual] epoch {ep:3d} | val_loss {vl:.4f} | best {best:.4f} | bad {bad}")
        if bad >= PATIENCE: print(f"[dual] early stop @ {ep}"); break
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(OUT_DIR,"dual_head_resnet18.pt"))
    return model

## 7. Decision policy, selective metrics, patient-level bootstrap

In [ ]:
def apply_policy(dec_probs, var, var_thr, theta_a=THETA_A, theta_r=THETA_R, delta=DELTA):
    pa, pr = dec_probs[:,ACCEPT], dec_probs[:,REVIEW]
    dec = np.full(len(dec_probs), REVIEW)
    acc = (pa > theta_a) & (pa > pr + delta); dec[acc] = ACCEPT
    dec[var <= var_thr] = DEFER
    return dec

def selective_metrics(y, pblast, dec, blast_idx=0):
    am = dec == ACCEPT
    out = dict(coverage=float(am.mean()), review_rate=float((dec==REVIEW).mean()),
               defer_rate=float((dec==DEFER).mean()), accepted_n=int(am.sum()))
    if am.sum()==0: out.update(acc_accuracy=np.nan,auc=np.nan,blast_sensitivity=np.nan,blast_fnr=np.nan); return out
    yt, pb = y[am], pblast[am]; yp = (pb>=0.5).astype(int); yb = (yt==blast_idx).astype(int)
    out["acc_accuracy"]=float(accuracy_score(yb,yp))
    out["auc"]=float(roc_auc_score(yb,pb)) if len(np.unique(yb))>1 else np.nan
    tp=int(((yp==1)&(yb==1)).sum()); fn=int(((yp==0)&(yb==1)).sum())
    out["blast_sensitivity"]=tp/(tp+fn+1e-9); out["blast_fnr"]=fn/(tp+fn+1e-9)
    return out

def bootstrap_patient(y, pblast, dec, pid, blast_idx=0, n_boot=N_BOOTSTRAP):
    pid = np.asarray(pid); uniq = np.unique(pid); rng = np.random.default_rng(SEED); s, f = [], []
    for _ in range(n_boot):
        chosen = rng.choice(uniq, len(uniq), replace=True)
        mask = np.concatenate([np.where(pid==p)[0] for p in chosen])
        m = selective_metrics(y[mask], pblast[mask], dec[mask], blast_idx)
        if not np.isnan(m["blast_sensitivity"]): s.append(m["blast_sensitivity"]); f.append(m["blast_fnr"])
    ci = lambda a: [float(np.percentile(a,2.5)), float(np.percentile(a,97.5))]
    return {"sensitivity_CI":ci(s), "fnr_CI":ci(f)}

def expected_calibration_error(probs, labels, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1); acc=(pred==labels).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0; table=[]
    for i in range(n_bins):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()==0: table.append((bins[i],bins[i+1],0,np.nan,np.nan)); continue
        a,c=acc[m].mean(),conf[m].mean(); ece+=(m.sum()/len(conf))*abs(a-c)
        table.append((bins[i],bins[i+1],int(m.sum()),float(a),float(c)))
    return ece, table

## 8. Matched-protocol baselines (Table 3)

In [ ]:
@torch.no_grad()
def mc_dropout_probs(model, loader, passes=20):
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train()
    P=[]
    for _ in range(passes):
        ps=[model(x.to(DEVICE)).softmax(1).cpu() for x,_ in loader]; P.append(torch.cat(ps))
    P=torch.stack(P); mean_p=P.mean(0).numpy()
    return mean_p, -(mean_p*np.log(mean_p+1e-12)).sum(1)

def selective_from_uncertainty(y, pblast, unc, pid, cap=0.35, blast_idx=0):
    best=None
    for q in np.linspace(0.05,0.95,19):
        thr=np.quantile(unc,q); dec=np.where(unc<=thr, ACCEPT, REVIEW)
        m=selective_metrics(y,pblast,dec,blast_idx)
        if m["review_rate"]<=cap and not np.isnan(m["blast_fnr"]):
            if best is None or m["blast_fnr"]<best[0]: best=(m["blast_fnr"],m,dec)
    if best is None:
        thr=np.quantile(unc,0.65); dec=np.where(unc<=thr,ACCEPT,REVIEW)
        best=(np.nan, selective_metrics(y,pblast,dec,blast_idx), dec)
    _,m,dec=best; m.update(bootstrap_patient(y,pblast,dec,pid,blast_idx)); return m

## 9. Decision-level Grad-CAM and attention metrics

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model=model; self.a=None; self.g=None
        layer.register_forward_hook(lambda m,i,o: setattr(self,"a",o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"g",go[0].detach()))
    def __call__(self, x, head="dec", target=ACCEPT):
        self.model.zero_grad(); cl, dl = self.model(x)
        (dl if head=="dec" else cl)[:,target].sum().backward()
        w=self.g.mean(dim=(2,3),keepdim=True); cam=F.relu((w*self.a).sum(1))
        cam=cam/(cam.amax(dim=(1,2),keepdim=True)+1e-8); return cam.cpu().numpy()

def cell_mask(x_unnorm):  # C-NMC images are pre-segmented; background ~0
    return (x_unnorm.mean(0) > 0.05).cpu().numpy()

def attention_metrics(cam, mask):
    eps=1e-8; p=cam/(cam.sum()+eps); entropy=float(-(p*np.log(p+eps)).sum())
    inside=cam[mask]; peakiness=float(cam.max()/(cam.mean()+1e-6))
    focus=float(inside.sum()/(cam.sum()+eps))
    H,W=cam.shape; ys,xs=np.mgrid[0:H,0:W]
    ccom=np.array([(cam*ys).sum(),(cam*xs).sum()])/(cam.sum()+eps)
    cm=mask.astype(float); mcom=np.array([(cm*ys).sum(),(cm*xs).sum()])/(cm.sum()+eps)
    return dict(entropy=entropy, peakiness=peakiness, focus_ratio=focus,
                com_distance=float(np.linalg.norm(ccom-mcom)/math.hypot(H,W)))

def cohens_d(a,b):
    a,b=np.asarray(a),np.asarray(b); na,nb=len(a),len(b)
    sp=math.sqrt(((na-1)*a.var(ddof=1)+(nb-1)*b.var(ddof=1))/(na+nb-2)+1e-12)
    return float((a.mean()-b.mean())/(sp+1e-12))

def compare_attention(acc, rev, k=4):
    r={}
    for key in acc:
        u,p=mannwhitneyu(acc[key],rev[key],alternative="two-sided")
        r[key]=dict(p_value=float(p),p_bonferroni=float(min(p*k,1.0)),cohens_d=cohens_d(acc[key],rev[key]),
                    accept_mean=float(np.mean(acc[key])),review_mean=float(np.mean(rev[key])))
    return r

## 10. Figure generation (Fig. 1, 4, 5)
Saves publication-quality PNGs (300 dpi) to `OUT_DIR/figures`. Figures 2 (architecture) and 3 (pipeline) are schematic diagrams drawn separately.

In [ ]:
def fig1_examples(samples, targets, classes, n=4):
    bi, ni = classes.index('blast'), classes.index('normal')
    bidx = np.where(targets==bi)[0][:n]; nidx = np.where(targets==ni)[0][:n]
    fig, ax = plt.subplots(2, n, figsize=(2.4*n, 5))
    from PIL import Image
    for j,i in enumerate(bidx):
        ax[0,j].imshow(Image.open(samples[i][0])); ax[0,j].axis('off')
        if j==0: ax[0,j].set_ylabel("Blast", fontsize=12)
    for j,i in enumerate(nidx):
        ax[1,j].imshow(Image.open(samples[i][0])); ax[1,j].axis('off')
        if j==0: ax[1,j].set_ylabel("Normal", fontsize=12)
    fig.suptitle("Representative C-NMC cell images", fontsize=13)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"fig1_examples.png"), dpi=300, bbox_inches='tight'); plt.close()
    print("saved fig1")

def fig5_reliability(table, ece):
    mids=[(a+b)/2 for a,b,n,acc,conf in table]; accs=[acc for *_,acc,_ in table]
    fig,axx=plt.subplots(figsize=(5,5))
    axx.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
    axx.bar(mids,[a if not np.isnan(a) else 0 for a in accs],width=0.09,alpha=0.8,edgecolor='black',label='Model')
    axx.set_xlabel('Confidence',fontsize=12); axx.set_ylabel('Accuracy',fontsize=12)
    axx.set_title(f'Reliability diagram (ECE = {ece:.3f})',fontsize=12)
    axx.legend(fontsize=10); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,"fig5_reliability.png"),dpi=300,bbox_inches='tight'); plt.close()
    print("saved fig5")

def fig4_gradcam(model, loader, cam_engine, var_thr, n=3):
    import matplotlib.cm as cmx
    mean_t=torch.tensor(NORM_MEAN).view(3,1,1); std_t=torch.tensor(NORM_STD).view(3,1,1)
    shown=0; fig,ax=plt.subplots(2,n,figsize=(2.6*n,5.2))
    for x,y in loader:
        dprob=model(x.to(DEVICE))[1].softmax(1).detach().cpu().numpy()
        dec=apply_policy(dprob, intensity_variance([(x,y)]), var_thr)
        cam=cam_engine(x.to(DEVICE),head="dec",target=ACCEPT)
        cam_rs=F.interpolate(torch.tensor(cam)[:,None],size=(IMG_SIZE,IMG_SIZE),mode="bilinear",align_corners=False)[:,0].numpy()
        for i in range(x.size(0)):
            if shown>=n: break
            raw=(x[i]*std_t+mean_t).permute(1,2,0).numpy().clip(0,1)
            ax[0,shown].imshow(raw); ax[0,shown].axis('off')
            ax[0,shown].set_title(["ACCEPT","REVIEW","DEFER"][dec[i]],fontsize=11)
            ax[1,shown].imshow(raw); ax[1,shown].imshow(cam_rs[i],cmap='jet',alpha=0.5); ax[1,shown].axis('off')
            shown+=1
        if shown>=n: break
    fig.suptitle("Decision-level Grad-CAM (top: input, bottom: attention)",fontsize=12)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"fig4_gradcam.png"),dpi=300,bbox_inches='tight'); plt.close()
    print("saved fig4")

## 11. Run everything

In [ ]:
def main():
    base, classes, samples, targets, groups, tr_i, va_i, te_i = build_patient_split(DATA_DIR)
    blast_idx = classes.index("blast")
    counts = np.bincount(targets[tr_i], minlength=len(classes)).astype(float)
    cw = torch.tensor(counts.sum()/(len(classes)*(counts+1e-9)), dtype=torch.float32)
    tr_loader, va_loader, te_loader = make_loaders(DATA_DIR, tr_i, va_i, te_i)

    print("\n== Baseline classifier ==")
    baseline = train_classifier(make_resnet18(), tr_loader, va_loader, cw, tag="baseline")

    print("\n== Decision pseudo-labels ==")
    dec_full = torch.full((len(samples),), -1, dtype=torch.long)
    tr_eval = DataLoader(Subset(datasets.ImageFolder(DATA_DIR, transform=eval_tfms), list(tr_i)), BATCH_SIZE, shuffle=False, num_workers=2)
    dec_tr, _ = build_decision_labels(tr_eval, baseline)
    for local, gi in enumerate(tr_i): dec_full[gi] = dec_tr[local]

    print("\n== Dual-head training ==")
    model = train_dual_head(DATA_DIR, tr_i, va_i, dec_full, cw)

    print("\n== Test evaluation ==")
    var_thr = np.percentile(intensity_variance(tr_eval), DEFER_VAR_PERCENTILE)
    model.eval(); cls_pb, dec_pr, y_all = [], [], []
    test_paths = [samples[i][0] for i in te_i]
    with torch.no_grad():
        for x,y in te_loader:
            cl,dl=model(x.to(DEVICE)); cls_pb.append(cl.softmax(1)[:,blast_idx].cpu().numpy())
            dec_pr.append(dl.softmax(1).cpu().numpy()); y_all.append(y.numpy())
    cls_pb=np.concatenate(cls_pb); dec_pr=np.concatenate(dec_pr); y_all=np.concatenate(y_all)
    test_var=intensity_variance(te_loader); pid=np.array([patient_id_from_path(p) for p in test_paths])

    # match baseline coverage by sweeping theta_a
    best_t,best_gap=THETA_A,1e9
    for t in np.linspace(0.30,0.90,25):
        cov=(apply_policy(dec_pr,test_var,var_thr,theta_a=t)==ACCEPT).mean()
        if abs(cov-TARGET_COVERAGE)<best_gap: best_gap,best_t=abs(cov-TARGET_COVERAGE),t
    print(f"matched theta_a={best_t:.3f}")
    decision=apply_policy(dec_pr,test_var,var_thr,theta_a=best_t)
    m=selective_metrics(y_all,cls_pb,decision,blast_idx); m.update(bootstrap_patient(y_all,cls_pb,decision,pid,blast_idx))
    print("PROPOSED:", json.dumps(m,indent=2,default=float))

    print("\n== Calibration ==")
    tl, ty = predict_logits(baseline, te_loader); probs=tl.softmax(1).numpy()
    ece, table = expected_calibration_error(probs, ty.numpy()); print(f"ECE={ece:.4f}")

    print("\n== Baselines (Table 3) ==")
    t3={"Proposed":m}
    bl,_=predict_logits(baseline,te_loader); bp=bl.softmax(1).numpy()
    t3["ConfidenceThresholding"]=selective_from_uncertainty(y_all,bp[:,blast_idx],1-bp.max(1),pid,blast_idx=blast_idx)
    mc=train_classifier(make_resnet18(dropout=0.5),tr_loader,va_loader,cw,tag="mcdrop")
    mcp,mcu=mc_dropout_probs(mc,te_loader,20)
    t3["MC_Dropout"]=selective_from_uncertainty(y_all,mcp[:,blast_idx],mcu,pid,blast_idx=blast_idx)
    ep=[]
    for s in range(3):
        torch.manual_seed(SEED+s); em=train_classifier(make_resnet18(),tr_loader,va_loader,cw,tag=f"ens{s}")
        ep.append(predict_logits(em,te_loader)[0].softmax(1).numpy())
    ens=np.stack(ep); t3["DeepEnsemble"]=selective_from_uncertainty(y_all,ens.mean(0)[:,blast_idx],ens.std(0).mean(1),pid,blast_idx=blast_idx)
    json.dump(t3,open(os.path.join(OUT_DIR,"table3_comparison.json"),"w"),indent=2,default=float)

    print("\n== Attention metrics ==")
    cam_engine=GradCAM(model, model.backbone.layer4[-1])
    acc={k:[] for k in ["entropy","peakiness","focus_ratio","com_distance"]}; rev={k:[] for k in acc}
    mean_t=torch.tensor(NORM_MEAN).view(3,1,1); std_t=torch.tensor(NORM_STD).view(3,1,1); seen=0
    for x,y in te_loader:
        dprob=model(x.to(DEVICE))[1].softmax(1).detach().cpu().numpy()
        dec_b=apply_policy(dprob,intensity_variance([(x,y)]),var_thr)
        cam=cam_engine(x.to(DEVICE),head="dec",target=ACCEPT)
        cam_rs=F.interpolate(torch.tensor(cam)[:,None],size=(IMG_SIZE,IMG_SIZE),mode="bilinear",align_corners=False)[:,0].numpy()
        for i in range(x.size(0)):
            mt=attention_metrics(cam_rs[i], cell_mask(x[i]*std_t+mean_t))
            tgt=acc if dec_b[i]==ACCEPT else (rev if dec_b[i]==REVIEW else None)
            if tgt is not None:
                for k in mt: tgt[k].append(mt[k])
        seen+=x.size(0)
        if seen>=1000: break
    att=compare_attention(acc,rev)
    json.dump({"accept_n":len(acc["entropy"]),"review_n":len(rev["entropy"]),"tests":att},
              open(os.path.join(OUT_DIR,"attention_metrics.json"),"w"),indent=2,default=float)
    print(json.dumps(att,indent=2,default=float))

    print("\n== Figures ==")
    fig1_examples(samples,targets,classes); fig5_reliability(table,ece)
    fig4_gradcam(model,te_loader,cam_engine,var_thr)

    summary={"proposed_test":m,"ECE":ece,"attention_tests":att,
             "split":{"patients":len(set(groups)),"train":len(tr_i),"val":len(va_i),"test":len(te_i)}}
    json.dump(summary,open(os.path.join(OUT_DIR,"RESULTS_SUMMARY.json"),"w"),indent=2,default=float)
    print("\n>>> Done. Results + figures written to", OUT_DIR)

main()